# P1 — Data Preprocessing

PCRR: run stages in order, with the same MODE and BANK_EXTENSION. No P0.
Start SMOKE, then PILOT, then FULL. P5 opens Test **only in FULL after P4 freeze**.
SMOKE/PILOT P5 exports development results; it does not mark P5 PASS.
Read `README.md`, `reports/METHODOLOGY.md` and the generated acceptance report.
Data and checkpoints remain on Drive. Restart runtime after first dependency installation.


In [1]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/FJCAI2027_PCRR').resolve()
assert (ROOT / 'data/raw/dataset_manifest.json').is_file(), 'Missing existing project/manifest'
assert (ROOT / 'run_all.py').is_file(), 'Wait for project files to sync to Drive'
probe = ROOT / '.colab_write_probe'
probe.write_text('ok'); probe.unlink()
print(ROOT)


Mounted at /content/drive
/content/drive/MyDrive/FJCAI2027_PCRR


In [2]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT/'requirements.txt')], check=True)
print('After initial install, restart runtime, then rerun mount and remaining cells.')


After initial install, restart runtime, then rerun mount and remaining cells.


In [3]:
MODE = 'FULL'  # SMOKE -> PILOT -> FULL
BANK_EXTENSION = True  # Set True consistently for all FULL stages to include Bank
import os
os.environ['PCRR_ROOT'] = str(ROOT)
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'


In [4]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pytest', '-q', str(ROOT/'tests/test_core.py')], cwd=ROOT, check=True)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pytest', '-q', '/content/drive/MyDrive/FJCAI2027_PCRR/tests/test_core.py'], returncode=0)

In [5]:
command = [sys.executable, str(ROOT/'run_all.py'), '--root', str(ROOT), '--mode', MODE, '--stage', 'P1']
if BANK_EXTENSION: command.append('--bank')
result = subprocess.run(command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(result.stdout)
result.check_returncode()


Experiment: full-1c89a647ec3466da
P1: starting
P1: PASS
Artifacts: /content/drive/MyDrive/FJCAI2027_PCRR/results/full-1c89a647ec3466da



In [6]:
from IPython.display import Markdown, display
report = ROOT/'reports/P1_ACCEPTANCE.md'
if report.exists(): display(Markdown(report.read_text()))
print('Detailed artifacts and status are under results/<experiment-id>/ and reports/<experiment-id>/.')


# P1 Acceptance — full-1c89a647ec3466da

Status: **PASS**

## Work and evidence

All three raw datasets passed hashes, shapes and labels. Split disjointness and train-only transformer reload checks passed. Test was partitioned and persisted, not evaluated.

## Artifacts

- results/full-1c89a647ec3466da/
- data/processed/full-1c89a647ec3466da/
- checkpoints/full-1c89a647ec3466da/

## Verification

`python -m pytest -q`

Stage command: `python run_all.py --mode FULL --stage P1`

Stage PASS refers to executed runtime assertions; unit-test evidence is recorded separately in reports/IMPLEMENTATION.md.

## Limits and next gate

SMOKE/PILOT are development-only, not publication evidence. P5 requires FULL and verified P4 artifacts. Scientific acceptance requires inspection of actual results.


Detailed artifacts and status are under results/<experiment-id>/ and reports/<experiment-id>/.
